<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/07_AED_artefatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB07 — Análise Exploratória dos Artefatos do Pipeline (AED)

## 1. Contexto

Este notebook realiza a Análise Exploratória de Dados (AED) sobre os artefatos
produzidos pelos notebooks NB01 a NB06 do pipeline PPCOMP_DM. Ele não lê o
arquivo CSV bruto (`borg_traces_data.csv`) diretamente, nem envolve etapas de
modelagem (NB08/NB09).

A separação entre a AED do dado bruto (NB00) e a AED dos artefatos do pipeline
(NB07) é uma decisão metodológica deliberada: o NB00 documenta a fonte primária
tal como recebida; o NB07 documenta o que o pipeline faz com essa fonte, de modo
que as decisões de NB08 possam ser justificadas com base no estado dos dados
após o processamento.

## 2. Objetivo

- Verificar a integridade estrutural de cada artefato intermediário (shape,
  colunas, missingness);
- Comparar estatísticas-chave com os valores de referência do NB00 (dado bruto);
- Documentar o efeito do NB02 sobre o bloco inicial (`hour == 0`);
- Analisar a série contínua de janelas de 5 minutos (NB03);
- Descrever os parâmetros de criticidade calculados no NB04 (µ, σ, limiar µ+2σ)
  e o perfil dos episódios detectados;
- Caracterizar as features engenheiradas pelo NB05 (distribuições, lags,
  rolling stats);
- Avaliar a distribuição dos estados operacionais (NB06: `window_5min_labeled`)
  e o desbalanceamento do dataset supervisionado
  (`anticipation_supervised_dataset`, cenário `primary_K24_H12`);
- Construir o fluxo de linhas NB00 → NB06 e persistir resumo JSON
  (`07_aed_pipeline_summary.json`) e figuras exploratórias.

## 3. Papel no Pipeline

Este notebook é **descritivo e dependente exclusivamente de NB01–NB06**. Não
transforma dados, não treina modelos e não produz artefatos consumidos por
NB08/NB09. Serve de elo de auditoria entre a preparação dos dados e a
modelagem.

## 4. Artefatos Utilizados

| Artefato | Origin NB | Path base | Descrição |
|---|---|---|---|
| `trace_raw_validated.parquet` | NB01 | PROCESSED_PATH | Dataset validado e tipado |
| `01_ingest_validate_summary.json` | NB01 | REPORTS_PATH | Resumo NB01 |
| `google_trace_clean.parquet` | NB02 | PROCESSED_PATH | Dataset limpo/normalizado |
| `02_clean_normalize_summary.json` | NB02 | REPORTS_PATH | Resumo NB02 |
| `window_5min_series.parquet` | NB03 | FEATURES_PATH | Série contínua de janelas 5 min |
| `window_5min_base.parquet` | NB03 | FEATURES_PATH | Base agregada (buckets observados) |
| `03_window_5min_base_summary.json` | NB03 | REPORTS_PATH | Resumo NB03 |
| `window_5min_series_scored.parquet` | NB04 | FEATURES_PATH | Série com `fail_rate` e `is_critical` |
| `episodes_detected.parquet` | NB04 | FEATURES_PATH | Episódios críticos detectados |
| `04_detect_episodes_summary.json` | NB04 | REPORTS_PATH | Resumo NB04 (µ, σ, limiar) |
| `window_5min_features.parquet` | NB05 | FEATURES_PATH | Série com features engenheiradas |
| `05_feature_engineering_summary.json` | NB05 | REPORTS_PATH | Resumo NB05 |
| `window_5min_labeled.parquet` | NB06 | FEATURES_PATH | Série com coluna `state` |
| `anticipation_supervised_dataset.parquet` | NB06 | FEATURES_PATH | Dataset supervisionado (K=24, H=12) |
| `06_labeling_states_summary.json` | NB06 | REPORTS_PATH | Resumo NB06 |
| `00_aed_raw_summary.json` | NB00 | REPORTS_PATH | Referência do dado bruto |

## 5. Estrutura do Notebook

- **Célula 1:** Bootstrap, imports, configuração de paths, funções auxiliares.
- **Célula 2:** AED NB01 + NB02 — estrutura, limpeza, tratamento do `hour == 0`.
- **Célula 3:** AED NB03 — série de 5 min, cobertura temporal, bucket 0.
- **Célula 4:** AED NB04 — parâmetros µ/σ/limiar, críticos, episódios.
- **Célula 5:** AED NB05 — features engenheiradas, distribuições, correlações.
- **Célula 6:** AED NB06 — estados, dataset supervisionado, desbalanceamento.
- **Célula 7:** Fluxo de linhas NB00→NB06, consolidação e persistência.

In [3]:
# ============================================================
# NB07 — AED dos Artefatos do Pipeline (NB01–NB06)
# Pipeline PPCOMP_DM (Google Cluster Trace)
# Sem leitura do CSV bruto | Sem modelagem
# ============================================================

# ─────────────────────────────────────────────────────────────
# CÉLULA 1 — Bootstrap, imports e configuração
# ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
import subprocess
import importlib
from pathlib import Path
from importlib.machinery import PathFinder

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

repo_str = str(REPO_DIR)
if repo_str in sys.path:
    sys.path.remove(repo_str)
sys.path.insert(0, repo_str)
importlib.invalidate_caches()
if PathFinder not in sys.meta_path:
    sys.meta_path.append(PathFinder)

(REPO_DIR / "src").mkdir(parents=True, exist_ok=True)
init_file = REPO_DIR / "src" / "__init__.py"
if not init_file.exists():
    init_file.write_text("# src package\n", encoding="utf-8")

# ── Imports ──────────────────────────────────────────────────
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display
from importlib import reload

import src.paths as _paths
reload(_paths)
from src.paths import (
    RAW_PATH, PROCESSED_PATH, FEATURES_PATH,
    REPORTS_PATH, ensure_dirs,
)
ensure_dirs()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Helpers ───────────────────────────────────────────────────
def log(msg: str) -> None:
    print(f"[NB07_AED_PIPELINE] {msg}")

FIG_DIR = REPORTS_PATH / "figures_nb07_aed_pipeline"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name: str) -> str:
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    log(f"Figura salva: {out}")
    return str(out)

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

# ── Paths dos artefatos ───────────────────────────────────────
# NB01
NB01_PARQUET  = PROCESSED_PATH / "trace_raw_validated.parquet"
NB01_JSON     = REPORTS_PATH   / "01_ingest_validate_summary.json"
# NB02
NB02_PARQUET  = PROCESSED_PATH / "google_trace_clean.parquet"
NB02_JSON     = REPORTS_PATH   / "02_clean_normalize_summary.json"
# NB03
NB03_SERIES   = FEATURES_PATH  / "window_5min_series.parquet"
NB03_BASE     = FEATURES_PATH  / "window_5min_base.parquet"
NB03_JSON     = REPORTS_PATH   / "03_window_5min_base_summary.json"
# NB04
NB04_SCORED   = FEATURES_PATH  / "window_5min_series_scored.parquet"
NB04_EPISODES = FEATURES_PATH  / "episodes_detected.parquet"
NB04_JSON     = REPORTS_PATH   / "04_detect_episodes_summary.json"
# NB05
NB05_FEATURES = FEATURES_PATH  / "window_5min_features.parquet"
NB05_JSON     = REPORTS_PATH   / "05_feature_engineering_summary.json"
# NB06
NB06_LABELED  = FEATURES_PATH  / "window_5min_labeled.parquet"
NB06_SUPERV   = FEATURES_PATH  / "anticipation_supervised_dataset.parquet"
NB06_JSON     = REPORTS_PATH   / "06_labeling_states_summary.json"
# NB00 referência
NB00_JSON     = REPORTS_PATH   / "00_aed_raw_summary.json"
# Saída NB07
SUMMARY_FILE  = REPORTS_PATH   / "07_aed_pipeline_summary.json"

# ── Referência NB00 ───────────────────────────────────────────
raw_ref = {}
if NB00_JSON.exists():
    raw_ref = json.loads(NB00_JSON.read_text(encoding="utf-8"))
    log(f"Referência NB00 carregada: {NB00_JSON}")
else:
    log("AVISO: 00_aed_raw_summary.json não encontrado — comparativos NB00 omitidos.")

summary = {
    "notebook": "NB07",
    "purpose": "aed_pipeline_artifacts_nb01_to_nb06",
    "uses_downstream_artifacts": True,
    "nb00_reference_loaded": bool(raw_ref),
}

log(f"PROCESSED_PATH = {PROCESSED_PATH}")
log(f"FEATURES_PATH  = {FEATURES_PATH}")
log(f"REPORTS_PATH   = {REPORTS_PATH}")
log(f"FIG_DIR        = {FIG_DIR}")
print("\n[NB07] Célula 1 — Bootstrap concluído.")

# ─────────────────────────────────────────────────────────────
# CÉLULA 2 — AED NB01 e NB02: dados validados e limpos
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB01 — DATASET VALIDADO (trace_raw_validated.parquet)")
print("=" * 60)

assert NB01_PARQUET.exists(), f"Artefato não encontrado: {NB01_PARQUET}"
df01 = pd.read_parquet(NB01_PARQUET)
log(f"NB01 lido: {df01.shape[0]} linhas x {df01.shape[1]} colunas")

print(f"\n[Shape] {df01.shape}")
print("\n[Tipos de dados]")
display(df01.dtypes.to_frame("dtype"))

# ── Missingness NB01 ─────────────────────────────────────────
miss01 = df01.isna().sum()
miss01_pct = (df01.isna().mean() * 100).round(2)
miss_df01 = pd.DataFrame({"missing_count": miss01, "missing_pct": miss01_pct})
miss_df01 = miss_df01[miss_df01["missing_count"] > 0].sort_values(
    "missing_pct", ascending=False
)
print("\n[Missingness NB01]")
display(miss_df01 if len(miss_df01) > 0 else pd.DataFrame({"resultado": ["Sem NaN"]}))

# ── Comparativo NB00 → NB01 ──────────────────────────────────
raw_rows = raw_ref.get("raw_rows", None) if raw_ref else None
delta01 = df01.shape[0] - raw_rows if raw_rows else None
if raw_rows:
    print(f"\n[Comparativo NB00 → NB01]")
    print(f"  Linhas brutas (NB00)   : {raw_rows:,}")
    print(f"  Linhas validadas (NB01): {df01.shape[0]:,}")
    print(f"  Diferença              : {delta01:+,} ({delta01 / raw_rows * 100:+.3f}%)")

# ── Resumo NB01 JSON ─────────────────────────────────────────
nb01_meta = {}
if NB01_JSON.exists():
    nb01_meta = json.loads(NB01_JSON.read_text(encoding="utf-8"))
    log("NB01 summary JSON carregado.")

summary["nb01"] = {
    "rows": int(df01.shape[0]),
    "cols": int(df01.shape[1]),
    "columns": list(df01.columns),
    "missing_by_column": {c: int(v) for c, v in miss01.items() if v > 0},
    "delta_rows_vs_raw": delta01,
}

# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB02 — DATASET LIMPO (google_trace_clean.parquet)")
print("=" * 60)

assert NB02_PARQUET.exists(), f"Artefato não encontrado: {NB02_PARQUET}"
df02 = pd.read_parquet(NB02_PARQUET)
log(f"NB02 lido: {df02.shape[0]} linhas x {df02.shape[1]} colunas")

print(f"\n[Shape] {df02.shape}")
print(f"\n[Colunas adicionadas pelo NB02 vs NB01]")
new_cols = sorted(set(df02.columns) - set(df01.columns))
print("  " + str(new_cols))

# ── Resumo NB02 JSON ─────────────────────────────────────────
nb02_meta = {}
if NB02_JSON.exists():
    nb02_meta = json.loads(NB02_JSON.read_text(encoding="utf-8"))
    log("NB02 summary JSON carregado.")

rows_h0 = nb02_meta.get("rows_hour0", None)
rows_out = nb02_meta.get("rows_out", None)
scenario = nb02_meta.get("scenario_label", "—")
remove_h0 = nb02_meta.get("remove_hour_zero", None)

print(f"\n[Decisão sobre hour == 0 (NB02)]")
print(f"  scenario_label    : {scenario}")
print(f"  remove_hour_zero  : {remove_h0}")
print(f"  rows_hour0        : {rows_h0:,}" if rows_h0 else "  rows_hour0 : n/a")
print(f"  rows_out          : {rows_out:,}" if rows_out else "  rows_out   : n/a")

# ── Distribuição temporal NB02 ────────────────────────────────
if "hour" in df02.columns:
    HOUR_US = 3_600_000_000
    hourly = df02.groupby("hour").size()
    print(f"\n[Cobertura temporal NB02]")
    print(f"  hour range : {int(df02['hour'].min())}..{int(df02['hour'].max())}")
    print(f"  vol min/max/mean por hora : {hourly.min():,} / {hourly.max():,} / {hourly.mean():.0f}")

    plt.figure(figsize=(12, 4))
    plt.plot(hourly.index, hourly.values, linewidth=0.8)
    plt.xlabel("hour (relativo ao início)")
    plt.ylabel("n_registros")
    plt.title("NB07 — NB02: Volume de registros por hora")
    save_fig("fig_01_nb02_volume_por_hora.png")

summary["nb02"] = {
    "rows": int(df02.shape[0]),
    "cols": int(df02.shape[1]),
    "columns": list(df02.columns),
    "new_columns_vs_nb01": new_cols,
    "scenario_label": scenario,
    "remove_hour_zero": remove_h0,
    "rows_hour0": rows_h0,
}

del df01, df02  # liberar memória
print("\n[NB07] Célula 2 — NB01/NB02 concluída.")


# ─────────────────────────────────────────────────────────────
# CÉLULA 3 — AED NB03: série de janelas de 5 minutos
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB03 — SÉRIE DE JANELAS 5 MIN (window_5min_series.parquet)")
print("=" * 60)

assert NB03_SERIES.exists(), f"Artefato não encontrado: {NB03_SERIES}"
df03s = pd.read_parquet(NB03_SERIES)
log(f"NB03 série lida: {df03s.shape[0]} linhas x {df03s.shape[1]} colunas")

print(f"\n[Shape] {df03s.shape}")
print(f"\n[Colunas] {list(df03s.shape)}")
print("\n[Colunas]", list(df03s.columns))
print("\n[Amostra]")
display(df03s.head())
print("\n[Estatísticas descritivas]")
display(df03s.describe().T)

# ── Cobertura e gaps ──────────────────────────────────────────
bucket_min = int(df03s["bucket_id"].min())
bucket_max = int(df03s["bucket_id"].max())
n_buckets  = int(df03s["bucket_id"].nunique())
n_expected = bucket_max - bucket_min + 1
n_gaps     = n_expected - n_buckets

print(f"\n[Cobertura de buckets]")
print(f"  bucket_id range : {bucket_min}..{bucket_max}")
print(f"  buckets únicos  : {n_buckets}")
print(f"  gaps (janelas vazias) : {n_gaps}")

# Bucket 0 (bloco inicial)
if (df03s["bucket_id"] == 0).any():
    b0 = df03s.loc[df03s["bucket_id"] == 0].iloc[0]
    print(f"\n[Bucket 0 — bloco inicial]")
    for col in ["n_events", "n_failed", "n_machines", "n_collections", "mean_priority"]:
        if col in df03s.columns:
            print(f"  {col}: {b0[col]}")

# ── Série temporal: n_events ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df03s["bucket_id"], df03s["n_events"], linewidth=0.6)
axes[0].set_ylabel("n_events")
axes[0].set_title("NB07 — NB03: n_events por janela de 5 min")
axes[1].plot(df03s["bucket_id"], df03s["n_failed"], linewidth=0.6, color="tomato")
axes[1].set_ylabel("n_failed")
axes[1].set_title("NB07 — NB03: n_failed por janela de 5 min")
axes[1].set_xlabel("bucket_id")
save_fig("fig_02_nb03_series_eventos.png")

# ── Base vs Série ─────────────────────────────────────────────
if NB03_BASE.exists():
    df03b = pd.read_parquet(NB03_BASE)
    log(f"NB03 base lida: {df03b.shape[0]} linhas x {df03b.shape[1]} colunas")
    print(f"\n[Base agregada] shape: {df03b.shape}")
    print(f"  Buckets com eventos: {df03b.shape[0]}")
    print(f"  Buckets da série total (com gaps): {df03s.shape[0]}")
    del df03b

nb03_meta = {}
if NB03_JSON.exists():
    nb03_meta = json.loads(NB03_JSON.read_text(encoding="utf-8"))
    log("NB03 summary JSON carregado.")

summary["nb03"] = {
    "series_rows": int(df03s.shape[0]),
    "cols": int(df03s.shape[1]),
    "columns": list(df03s.columns),
    "bucket_min": bucket_min,
    "bucket_max": bucket_max,
    "n_gap_buckets": n_gaps,
    "n_events_total": int(df03s["n_events"].sum()),
    "n_failed_total": int(df03s["n_failed"].sum()),
    "avg_events_per_bucket": float(df03s["n_events"].mean()),
}

del df03s
print("\n[NB07] Célula 3 — NB03 concluída.")


# ─────────────────────────────────────────────────────────────
# CÉLULA 4 — AED NB04: criticidade e episódios
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB04 — CRITICIDADE E EPISÓDIOS")
print("=" * 60)

# ── Parâmetros do NB04 (JSON) ─────────────────────────────────
assert NB04_JSON.exists(), f"Artefato não encontrado: {NB04_JSON}"
nb04_meta = json.loads(NB04_JSON.read_text(encoding="utf-8"))
log(f"NB04 summary JSON carregado.")

# Extração dos parâmetros do cenário oficial (global)
global_info = nb04_meta.get("global", nb04_meta)  # adapta se estrutura diferir
mu04    = float(global_info.get("mu",        global_info.get("fail_rate_mu", 0.0)))
sigma04 = float(global_info.get("sigma",     global_info.get("fail_rate_sigma", 0.0)))
thr04   = float(global_info.get("threshold", global_info.get("threshold_mu_2sigma", mu04 + 2 * sigma04)))
n_critical = int(global_info.get("critical_windows", global_info.get("n_critical_windows", 0)))
n_episodes = int(global_info.get("episodes_detected", global_info.get("n_episodes", 0)))

print(f"\n[Parâmetros NB04 — limiar oficial (série global)]")
print(f"  µ              : {mu04:.6f}")
print(f"  σ              : {sigma04:.6f}")
print(f"  Limiar µ+2σ    : {thr04:.6f}")
print(f"  Janelas críticas: {n_critical}")
print(f"  Episódios       : {n_episodes}")

# ── Série scored ──────────────────────────────────────────────
assert NB04_SCORED.exists(), f"Artefato não encontrado: {NB04_SCORED}"
df04s = pd.read_parquet(NB04_SCORED)
log(f"NB04 série scored lida: {df04s.shape}")

print(f"\n[Colunas NB04 scored] {list(df04s.columns)}")
print("\n[Distribuição is_critical]")
display(df04s["is_critical"].value_counts(dropna=False).to_frame("count"))

# ── fail_rate ao longo do tempo ───────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(df04s["bucket_id"], df04s["fail_rate"], linewidth=0.6, label="fail_rate")
axes[0].axhline(thr04, linestyle="--", color="red", linewidth=1.2,
                label=f"µ+2σ = {thr04:.4f}")
axes[0].set_ylabel("fail_rate")
axes[0].set_title("NB07 — NB04: fail_rate por janela com limiar µ+2σ")
axes[0].legend(fontsize=9)

crit_vals = df04s["is_critical"].values
axes[1].fill_between(df04s["bucket_id"], crit_vals,
                     step="mid", alpha=0.7, color="tomato", label="is_critical=1")
axes[1].set_ylabel("is_critical")
axes[1].set_xlabel("bucket_id")
axes[1].set_title("NB07 — NB04: Janelas críticas ao longo do tempo")
axes[1].legend(fontsize=9)
save_fig("fig_03_nb04_fail_rate_e_criticos.png")

# Histograma de fail_rate
plt.figure(figsize=(9, 4))
plt.hist(df04s["fail_rate"], bins=50, edgecolor="none")
plt.axvline(thr04, linestyle="--", color="red", label=f"µ+2σ = {thr04:.4f}")
plt.axvline(mu04,  linestyle=":",  color="steelblue", label=f"µ = {mu04:.4f}")
plt.xlabel("fail_rate")
plt.ylabel("frequência")
plt.title("NB07 — NB04: Distribuição de fail_rate")
plt.legend(fontsize=9)
save_fig("fig_04_nb04_fail_rate_hist.png")

# Comparativo NB00 exploratório vs NB04 oficial
nb00_series = raw_ref.get("series_exploratory", {}) if raw_ref else {}
nb00_mu  = nb00_series.get("fail_rate_mean", None)
nb00_thr = nb00_series.get("threshold_mu_2sigma_exploratory", None)
if nb00_mu and mu04:
    print(f"\n[Comparativo NB00 exploratório → NB04 oficial]")
    print(f"  µ NB00  : {nb00_mu:.6f}   µ NB04  : {mu04:.6f}   Δ={mu04-nb00_mu:+.6f}")
    print(f"  thr NB00: {nb00_thr:.6f}   thr NB04: {thr04:.6f}   Δ={thr04-nb00_thr:+.6f}")

# ── Episódios ─────────────────────────────────────────────────
assert NB04_EPISODES.exists(), f"Artefato não encontrado: {NB04_EPISODES}"
df04e = pd.read_parquet(NB04_EPISODES)
log(f"NB04 episódios lidos: {df04e.shape}")

print(f"\n[Episódios NB04] shape: {df04e.shape}")
print(f"\n[Colunas] {list(df04e.columns)}")
print("\n[Estatísticas de duração dos episódios (windows)]")
if "duration_windows" in df04e.columns:
    display(df04e["duration_windows"].describe().to_frame("duration_windows"))
    plt.figure(figsize=(9, 4))
    plt.hist(df04e["duration_windows"], bins=30, edgecolor="none")
    plt.xlabel("duration_windows (janelas de 5 min)")
    plt.ylabel("frequência")
    plt.title("NB07 — NB04: Distribuição da duração dos episódios críticos")
    save_fig("fig_05_nb04_episode_duration.png")

summary["nb04"] = {
    "mu": mu04,
    "sigma": sigma04,
    "threshold_mu_2sigma": thr04,
    "n_critical_windows": n_critical,
    "n_episodes": n_episodes,
    "scored_rows": int(df04s.shape[0]),
    "scored_cols": int(df04s.shape[1]),
    "scored_columns": list(df04s.columns),
    "episodes_rows": int(df04e.shape[0]),
    "episodes_columns": list(df04e.columns),
    "nb00_exploratory_mu": nb00_mu,
    "nb00_exploratory_threshold": nb00_thr,
    "delta_mu_nb00_vs_nb04": round(mu04 - nb00_mu, 6) if nb00_mu else None,
    "delta_thr_nb00_vs_nb04": round(thr04 - nb00_thr, 6) if nb00_thr else None,
}

del df04s, df04e
print("\n[NB07] Célula 4 — NB04 concluída.")


# ─────────────────────────────────────────────────────────────
# CÉLULA 5 — AED NB05: features engenheiradas
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB05 — FEATURES ENGENHEIRADAS (window_5min_features.parquet)")
print("=" * 60)

assert NB05_FEATURES.exists(), f"Artefato não encontrado: {NB05_FEATURES}"
df05 = pd.read_parquet(NB05_FEATURES)
log(f"NB05 lido: {df05.shape[0]} linhas x {df05.shape[1]} colunas")

print(f"\n[Shape] {df05.shape}")
print(f"\n[Colunas] {list(df05.columns)}")

# ── Colunas novas vs NB04 (scored) ───────────────────────────
BASE_COLS_NB04 = {"bucket_id", "bucket_start_us", "n_events", "n_failed",
                  "n_machines", "n_collections", "mean_priority", "fail_rate",
                  "is_critical"}
engineered = sorted(set(df05.columns) - BASE_COLS_NB04)
print(f"\n[Features adicionadas pelo NB05] {engineered}")

# ── Missingness ───────────────────────────────────────────────
miss05 = df05.isna().sum()
miss_df05 = pd.DataFrame({
    "missing_count": miss05,
    "missing_pct": (df05.isna().mean() * 100).round(2),
})
miss_df05 = miss_df05[miss_df05["missing_count"] > 0].sort_values(
    "missing_pct", ascending=False
)
print("\n[Missingness NB05]")
display(miss_df05 if len(miss_df05) > 0 else pd.DataFrame({"resultado": ["Sem NaN"]}))

# ── Estatísticas descritivas ──────────────────────────────────
print("\n[Estatísticas descritivas — features numéricas]")
num_cols05 = df05.select_dtypes(include="number").columns.tolist()
display(df05[num_cols05].describe().T)

# ── Distribuições das features engenheiradas ──────────────────
eng_num = [c for c in engineered if df05[c].dtype.kind in "fi"][:12]
if eng_num:
    n = len(eng_num)
    ncols_fig = min(4, n)
    nrows_fig = (n + ncols_fig - 1) // ncols_fig
    fig, axes = plt.subplots(nrows_fig, ncols_fig, figsize=(16, 3.5 * nrows_fig))
    axes = np.array(axes).flatten()
    for i, col in enumerate(eng_num):
        axes[i].hist(df05[col].dropna(), bins=40, edgecolor="none")
        axes[i].set_title(col, fontsize=9)
        axes[i].tick_params(labelsize=7)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("NB07 — NB05: Distribuição das features engenheiradas", y=1.01)
    save_fig("fig_06_nb05_feature_distributions.png")

# ── Série temporal: lags e rolling ───────────────────────────
lag_roll_cols = [c for c in ["lag_1", "lag_2", "lag_3",
                              "rolling_mean_1h", "rolling_std_1h",
                              "pct_change", "zscore_expanding"]
                 if c in df05.columns]
if lag_roll_cols:
    fig, axes = plt.subplots(len(lag_roll_cols), 1,
                              figsize=(14, 2.5 * len(lag_roll_cols)),
                              sharex=True)
    if len(lag_roll_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, lag_roll_cols):
        ax.plot(df05["bucket_id"], df05[col], linewidth=0.5)
        ax.set_ylabel(col, fontsize=8)
        ax.tick_params(labelsize=7)
    axes[-1].set_xlabel("bucket_id")
    plt.suptitle("NB07 — NB05: Features temporais (lags e rolling)", y=1.01)
    save_fig("fig_07_nb05_temporal_features.png")

# ── Matriz de correlação ──────────────────────────────────────
corr_cols = [c for c in num_cols05
             if c not in ("bucket_id", "bucket_start_us")][:14]
if len(corr_cols) > 1:
    corr = df05[corr_cols].corr()
    sz = max(8, len(corr_cols) * 0.8)
    fig, ax = plt.subplots(figsize=(sz, sz * 0.85))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(corr_cols)))
    ax.set_yticks(range(len(corr_cols)))
    ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(corr_cols, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for r in range(len(corr_cols)):
        for c_idx in range(len(corr_cols)):
            val = corr.iloc[r, c_idx]
            ax.text(c_idx, r, f"{val:.2f}", ha="center", va="center",
                    fontsize=6, color="white" if abs(val) > 0.65 else "black")
    ax.set_title("NB07 — NB05: Correlação entre features")
    save_fig("fig_08_nb05_correlations.png")

nb05_meta = {}
if NB05_JSON.exists():
    nb05_meta = json.loads(NB05_JSON.read_text(encoding="utf-8"))

summary["nb05"] = {
    "rows": int(df05.shape[0]),
    "cols": int(df05.shape[1]),
    "columns": list(df05.columns),
    "engineered_features": engineered,
    "is_critical_ratio": float(df05["is_critical"].mean()) if "is_critical" in df05.columns else None,
    "missing_by_column": {c: int(v) for c, v in miss05.items() if v > 0},
}

del df05
print("\n[NB07] Célula 5 — NB05 concluída.")


# ─────────────────────────────────────────────────────────────
# CÉLULA 6 — AED NB06: estados e dataset supervisionado
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB06 — ESTADOS OPERACIONAIS (window_5min_labeled.parquet)")
print("=" * 60)

# ── window_5min_labeled ───────────────────────────────────────
assert NB06_LABELED.exists(), f"Artefato não encontrado: {NB06_LABELED}"
df06l = pd.read_parquet(NB06_LABELED)
log(f"NB06 labeled lido: {df06l.shape[0]} linhas x {df06l.shape[1]} colunas")

print(f"\n[Shape] {df06l.shape}")
print(f"\n[Colunas] {list(df06l.columns)}")

# Distribuição de estados
print("\n[Distribuição de 'state']")
state_vc = df06l["state"].value_counts(dropna=False)
state_pct = df06l["state"].value_counts(normalize=True, dropna=False).mul(100).round(2)
state_df = pd.DataFrame({"count": state_vc, "pct": state_pct})
display(state_df)

STATE_COLORS = {
    "NORMAL": "#5b9bd5",
    "BEFORE": "#f4a14e",
    "DURING": "#e74c3c",
    "AFTER":  "#82b74b",
}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
states_order = ["NORMAL", "BEFORE", "DURING", "AFTER"]
states_present = [s for s in states_order if s in state_vc.index]
counts_ordered = [state_vc.get(s, 0) for s in states_present]
colors_ordered = [STATE_COLORS.get(s, "gray") for s in states_present]
bars = axes[0].bar(states_present, counts_ordered, color=colors_ordered)
for bar, val in zip(bars, counts_ordered):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)
axes[0].set_title("NB07 — NB06: Distribuição de estados")
axes[0].set_ylabel("count")
axes[1].pie(counts_ordered,
            labels=[f"{s}\n({c:,})" for s, c in zip(states_present, counts_ordered)],
            colors=colors_ordered, autopct="%1.1f%%", startangle=90)
axes[1].set_title("NB07 — NB06: Proporção de estados")
save_fig("fig_09_nb06_state_distribution.png")

# Série temporal dos estados
if "bucket_id" in df06l.columns:
    state_num = df06l["state"].map({"NORMAL": 0, "BEFORE": 1, "DURING": 2, "AFTER": 3})
    plt.figure(figsize=(14, 3))
    plt.scatter(df06l["bucket_id"], state_num,
                c=[list(STATE_COLORS.values())[int(v)] if pd.notna(v) else "gray"
                   for v in state_num],
                s=2, alpha=0.8)
    plt.yticks([0, 1, 2, 3], ["NORMAL", "BEFORE", "DURING", "AFTER"])
    plt.xlabel("bucket_id")
    plt.title("NB07 — NB06: Estado ao longo do tempo")
    save_fig("fig_10_nb06_state_temporal.png")

# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB06 — DATASET SUPERVISIONADO (anticipation_supervised_dataset.parquet)")
print("=  cenário primary_K24_H12: K=24, H=12, horizonte=60 min  =")
print("=" * 60)

assert NB06_SUPERV.exists(), f"Artefato não encontrado: {NB06_SUPERV}"
df06s = pd.read_parquet(NB06_SUPERV)
log(f"NB06 supervisionado lido: {df06s.shape[0]} linhas x {df06s.shape[1]} colunas")

print(f"\n[Shape] {df06s.shape}")
print(f"\n[Colunas] {list(df06s.columns)}")

# Distribuição de transition_target
print("\n[Distribuição de 'transition_target']")
tgt_vc = df06s["transition_target"].value_counts(dropna=False).sort_index()
tgt_pct = df06s["transition_target"].value_counts(
    normalize=True, dropna=False).mul(100).round(2).sort_index()
display(pd.DataFrame({"count": tgt_vc, "pct": tgt_pct}))

n_pos = int((df06s["transition_target"] == 1).sum())
n_neg = int((df06s["transition_target"] == 0).sum())
ratio = n_pos / n_neg if n_neg > 0 else float("inf")
print(f"\n[Desbalanceamento]  positivo={n_pos:,}  negativo={n_neg:,}"
      f"  razão={ratio:.4f} ({n_pos/(n_pos+n_neg)*100:.1f}% positivo)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(["Negativo\n(0)", "Positivo\n(1)"], [n_neg, n_pos],
            color=["#5b9bd5", "#e74c3c"])
for i, val in enumerate([n_neg, n_pos]):
    axes[0].text(i, val + 50, f"{val:,}", ha="center", fontsize=10)
axes[0].set_title("NB07 — NB06: Desbalanceamento transition_target")
axes[0].set_ylabel("count")

axes[1].pie([n_neg, n_pos],
            labels=[f"Negativo ({n_neg:,})", f"Positivo ({n_pos:,})"],
            colors=["#5b9bd5", "#e74c3c"],
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("NB07 — NB06: Proporção de classes")
save_fig("fig_11_nb06_target_distribution.png")

# state × transition_target
if "state" in df06s.columns:
    ct = pd.crosstab(df06s["state"], df06s["transition_target"])
    print("\n[Tabela state × transition_target]")
    display(ct)

# Boxplot features × transition_target
feat_box = [c for c in ["fail_rate", "lag_1", "lag_2", "lag_3",
                          "rolling_mean_1h", "rolling_std_1h"]
            if c in df06s.columns]
if feat_box:
    n_box = len(feat_box)
    ncols_box = min(3, n_box)
    nrows_box = (n_box + ncols_box - 1) // ncols_box
    fig, axes = plt.subplots(nrows_box, ncols_box,
                              figsize=(14, 4.5 * nrows_box))
    axes = np.array(axes).flatten()
    for i, col in enumerate(feat_box):
        grp0 = df06s.loc[df06s["transition_target"] == 0, col].dropna()
        grp1 = df06s.loc[df06s["transition_target"] == 1, col].dropna()
        axes[i].boxplot([grp0, grp1], labels=["target=0", "target=1"],
                        patch_artist=True,
                        boxprops=dict(facecolor="lightblue"))
        axes[i].set_title(col, fontsize=9)
        axes[i].tick_params(labelsize=8)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("NB07 — NB06: Features por transition_target", y=1.01)
    save_fig("fig_12_nb06_features_by_target.png")

nb06_meta = {}
if NB06_JSON.exists():
    nb06_meta = json.loads(NB06_JSON.read_text(encoding="utf-8"))

summary["nb06"] = {
    "labeled_rows": int(df06l.shape[0]),
    "labeled_cols": int(df06l.shape[1]),
    "state_distribution": state_vc.to_dict(),
    "supervised_rows": int(df06s.shape[0]),
    "supervised_cols": int(df06s.shape[1]),
    "supervised_columns": list(df06s.columns),
    "scenario": "primary_K24_H12",
    "K": 24,
    "H": 12,
    "horizon_minutes": 60,
    "n_positive": n_pos,
    "n_negative": n_neg,
    "imbalance_ratio": round(ratio, 6),
    "positive_rate": round(n_pos / (n_pos + n_neg), 6),
}

del df06l, df06s
print("\n[NB07] Célula 6 — NB06 concluída.")

# ─────────────────────────────────────────────────────────────
# CÉLULA 7 — Fluxo de linhas, consolidação e persistência
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NB07 — FLUXO DE LINHAS NB00 → NB06 E CONSOLIDAÇÃO")
print("=" * 60)

# ── Fluxo de linhas ao longo do pipeline ─────────────────────
pipeline_rows = {
    "NB00\n(bruto CSV)":            raw_ref.get("raw_rows", None) if raw_ref else None,
    "NB01\n(validado)":             summary.get("nb01", {}).get("rows", None),
    "NB02\n(limpo)":                summary.get("nb02", {}).get("rows", None),
    "NB03\n(série 5min)":           summary.get("nb03", {}).get("series_rows", None),
    "NB05\n(features)":             summary.get("nb05", {}).get("rows", None),
    "NB06\n(rotulado)":             summary.get("nb06", {}).get("labeled_rows", None),
    "NB06\n(supervisionado\nK24H12)": summary.get("nb06", {}).get("supervised_rows", None),
}
valid_rows = {k: v for k, v in pipeline_rows.items() if v is not None}

print("\n[Fluxo de linhas ao longo do pipeline]")
display(
    pd.DataFrame(
        [(k.replace("\n", " "), f"{v:,}") for k, v in valid_rows.items()],
        columns=["Etapa", "Linhas"],
    )
)

fig, ax = plt.subplots(figsize=(12, 5))
etapas = list(valid_rows.keys())
valores = list(valid_rows.values())
bars = ax.bar(etapas, valores,
              color=["#2d6a9f", "#5b9bd5", "#5b9bd5", "#f4a14e",
                     "#f4a14e", "#82b74b", "#e74c3c"][:len(etapas)])
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(valores) * 0.01,
            f"{val:,}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Número de linhas")
ax.set_title("NB07 — Fluxo de registros ao longo do pipeline (NB00 → NB06)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.tick_params(axis="x", labelsize=8)
save_fig("fig_13_pipeline_row_flow.png")

# ── Achados principais ────────────────────────────────────────
findings = []
nb00_rows = raw_ref.get("raw_rows", None) if raw_ref else None
nb01_rows = summary.get("nb01", {}).get("rows", None)
if nb00_rows and nb01_rows:
    findings.append(
        f"NB01 preservou {nb01_rows:,} de {nb00_rows:,} registros do CSV bruto "
        f"({nb01_rows/nb00_rows*100:.2f}% de retenção)."
    )
nb02_info = summary.get("nb02", {})
if nb02_info:
    findings.append(
        f"NB02 manteve o bloco hour==0 (scenario_label='{nb02_info.get('scenario_label','?')}', "
        f"remove_hour_zero={nb02_info.get('remove_hour_zero','?')}), "
        f"preservando {nb02_info.get('rows_hour0', '?'):,} registros do bloco inicial."
        if isinstance(nb02_info.get("rows_hour0"), int)
        else f"NB02 aplicou cenário '{nb02_info.get('scenario_label','?')}'."
    )
nb03_info = summary.get("nb03", {})
if nb03_info:
    findings.append(
        f"NB03 gerou {nb03_info['series_rows']:,} janelas de 5 min "
        f"(bucket 0..{nb03_info['bucket_max']}), "
        f"com {nb03_info['n_gap_buckets']} janelas sem eventos."
    )
findings.append(
    f"NB04 calculou µ={summary['nb04']['mu']:.4f}, σ={summary['nb04']['sigma']:.4f}, "
    f"limiar µ+2σ={summary['nb04']['threshold_mu_2sigma']:.4f}; "
    f"{summary['nb04']['n_critical_windows']} janelas críticas e "
    f"{summary['nb04']['n_episodes']} episódios detectados."
)
nb05_info = summary.get("nb05", {})
if nb05_info:
    findings.append(
        f"NB05 engenheirou {len(nb05_info.get('engineered_features',[]))} features "
        f"({', '.join(nb05_info.get('engineered_features',[])[:4])}...), "
        f"resultando em {nb05_info['rows']:,} linhas × {nb05_info['cols']} colunas "
        f"após trim dos lags."
    )
nb06_info = summary.get("nb06", {})
if nb06_info:
    findings.append(
        f"NB06 (cenário primary_K24_H12, K=24, H=12, horizonte=60 min) gerou "
        f"{nb06_info['supervised_rows']:,} amostras supervisionadas, "
        f"com {nb06_info['n_positive']:,} positivos e {nb06_info['n_negative']:,} negativos "
        f"(razão={nb06_info['imbalance_ratio']:.4f})."
    )
nb04_nb00_delta = summary.get("nb04", {}).get("delta_thr_nb00_vs_nb04", None)
if nb04_nb00_delta is not None:
    findings.append(
        f"O limiar µ+2σ NB04 ({summary['nb04']['threshold_mu_2sigma']:.4f}) difere do "
        f"exploratório NB00 em {nb04_nb00_delta:+.6f}, confirmando convergência metodológica."
    )

print("\n[Achados exploratórios principais — NB07]")
for f in findings:
    print(f"  - {f}")

# ── Summary JSON final ────────────────────────────────────────
summary["findings"] = findings
summary["figures_dir"] = str(FIG_DIR)
summary["pipeline_row_flow"] = {k.replace("\n", " "): v for k, v in valid_rows.items()}
summary["outputs"] = {
    "summary_file": str(SUMMARY_FILE),
    "figures_dir": str(FIG_DIR),
}

SUMMARY_FILE.write_text(
    json.dumps(make_json_safe(summary), indent=2, ensure_ascii=False),
    encoding="utf-8",
)
log(f"Resumo salvo: {SUMMARY_FILE}")

print(f"\n=== RESUMO FINAL — NB07 ===")
print(f"Artefatos lidos  : NB01 a NB06")
print(f"Figuras geradas  : {FIG_DIR}")
print(f"Summary JSON     : {SUMMARY_FILE}")
print(f"\nImportante: este notebook não leu o CSV bruto nem artefatos de NB08/NB09.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[NB07_AED_PIPELINE] Referência NB00 carregada: /content/drive/MyDrive/Mestrado/04-reports/00_aed_raw_summary.json
[NB07_AED_PIPELINE] PROCESSED_PATH = /content/drive/MyDrive/Mestrado/02-datasets/02-processed
[NB07_AED_PIPELINE] FEATURES_PATH  = /content/drive/MyDrive/Mestrado/02-datasets/03-features
[NB07_AED_PIPELINE] REPORTS_PATH   = /content/drive/MyDrive/Mestrado/04-reports
[NB07_AED_PIPELINE] FIG_DIR        = /content/drive/MyDrive/Mestrado/04-reports/figures_nb07_aed_pipeline

[NB07] Célula 1 — Bootstrap concluído.

NB01 — DATASET VALIDADO (trace_raw_validated.parquet)
[NB07_AED_PIPELINE] NB01 lido: 405891 linhas x 21 colunas

[Shape] (405891, 21)

[Tipos de dados]


,dtype
time,int64
collection_id,Int64
scheduling_class,int64
priority,int64
instance_index,Int64
machine_id,Int64
resource_request,object
scheduler,float64
start_time,Int64
end_time,Int64



[Missingness NB01]


,missing_count,missing_pct
average_usage,405891,100.00
random_sample_usage,405891,100.00
maximum_usage,405891,100.00
cycles_per_instruction,124686,30.72
memory_accesses_per_instruction,124686,30.72
scheduler,959,0.24
resource_request,774,0.19



[Comparativo NB00 → NB01]
  Linhas brutas (NB00)   : 405,894
  Linhas validadas (NB01): 405,891
  Diferença              : -3 (-0.001%)
[NB07_AED_PIPELINE] NB01 summary JSON carregado.

NB02 — DATASET LIMPO (google_trace_clean.parquet)
[NB07_AED_PIPELINE] NB02 lido: 405891 linhas x 23 colunas

[Shape] (405891, 23)

[Colunas adicionadas pelo NB02 vs NB01]
  ['hour', 't_rel_us']
[NB07_AED_PIPELINE] NB02 summary JSON carregado.

[Decisão sobre hour == 0 (NB02)]
  scenario_label    : keep_hour0
  remove_hour_zero  : False
  rows_hour0        : 56,740
  rows_out          : 405,891

[Cobertura temporal NB02]
  hour range : 0..744
  vol min/max/mean por hora : 26 / 56,740 / 545
[NB07_AED_PIPELINE] Figura salva: /content/drive/MyDrive/Mestrado/04-reports/figures_nb07_aed_pipeline/fig_01_nb02_volume_por_hora.png

[NB07] Célula 2 — NB01/NB02 concluída.

NB03 — SÉRIE DE JANELAS 5 MIN (window_5min_series.parquet)
[NB07_AED_PIPELINE] NB03 série lida: 8930 linhas x 18 colunas

[Shape] (8930, 18)

[

...

transition_target,0,1
state,,
BEFORE,979,1987
NORMAL,4301,0


/tmp/ipykernel_18274/3862749344.py:678: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([grp0, grp1], labels=["target=0", "target=1"],
/tmp/ipykernel_18274/3862749344.py:678: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([grp0, grp1], labels=["target=0", "target=1"],
/tmp/ipykernel_18274/3862749344.py:678: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([grp0, grp1], labels=["target=0", "target=1"],
/tmp/ipykernel_18274/3862749344.py:678: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old

[NB07_AED_PIPELINE] Figura salva: /content/drive/MyDrive/Mestrado/04-reports/figures_nb07_aed_pipeline/fig_12_nb06_features_by_target.png

[NB07] Célula 6 — NB06 concluída.

NB07 — FLUXO DE LINHAS NB00 → NB06 E CONSOLIDAÇÃO

[Fluxo de linhas ao longo do pipeline]


,Etapa,Linhas
0,NB00 (bruto CSV),"405,894"
1,NB01 (validado),"405,891"
2,NB02 (limpo),"405,891"
3,NB03 (série 5min),"8,930"
4,NB05 (features),"8,927"
5,NB06 (rotulado),"8,927"
6,NB06 (supervisionado K24H12),"7,267"


[NB07_AED_PIPELINE] Figura salva: /content/drive/MyDrive/Mestrado/04-reports/figures_nb07_aed_pipeline/fig_13_pipeline_row_flow.png

[Achados exploratórios principais — NB07]
  - NB01 preservou 405,891 de 405,894 registros do CSV bruto (100.00% de retenção).
  - NB02 manteve o bloco hour==0 (scenario_label='keep_hour0', remove_hour_zero=False), preservando 56,740 registros do bloco inicial.
  - NB03 gerou 8,930 janelas de 5 min (bucket 0..8929), com 0 janelas sem eventos.
  - NB04 calculou µ=0.0000, σ=0.0000, limiar µ+2σ=0.0000; 0 janelas críticas e 0 episódios detectados.
  - NB05 engenheirou 19 features (event_ENABLE_count, event_EVICT_count, event_FAIL_count, event_FINISH_count...), resultando em 8,927 linhas × 28 colunas após trim dos lags.
  - NB06 (cenário primary_K24_H12, K=24, H=12, horizonte=60 min) gerou 7,267 amostras supervisionadas, com 1,987 positivos e 5,280 negativos (razão=0.3763).
[NB07_AED_PIPELINE] Resumo salvo: /content/drive/MyDrive/Mestrado/04-reports/07_aed_pipe